In [1]:
import pandas as pd

In [3]:
raw = pd.read_csv(r"C:\Users\USER\Downloads\teaching_prices_messy.csv")
raw

,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
...,...,...,...,...,...,...
92,2026-03-19,Pulse C,City 1,207.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
93,2026-03-19,Pulse C,City 2,213.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
94,2026-03-19,Sweetener D,City 1,178.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
95,2026-03-19,Sweetener D,City 2,184.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


In [4]:
print(raw.shape)
print(raw.isna().sum())
raw.info()

(97, 6)
date          0
item          0
city          0
price         1
unit          0
data_label    0
dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   date        97 non-null     str  
 1   item        97 non-null     str  
 2   city        97 non-null     str  
 3   price       96 non-null     str  
 4   unit        97 non-null     str  
 5   data_label  97 non-null     str  
dtypes: str(6)
memory usage: 15.1 KB


In [5]:
print("exact duplicates:", raw.duplicated().sum())
raw[raw.duplicated(keep=False)]

exact duplicates: 1


,date,item,city,price,unit,data_label
8,2026-01-08,Staple A,City 1,99.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
96,2026-01-08,Staple A,City 1,99.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


In [6]:
print(raw["item"].value_counts())
print(raw["city"].value_counts())
print(raw["date"].nunique())
print(raw.nunique())

item
Staple A       24
Oil B          24
Pulse C        24
Sweetener D    24
oil b           1
Name: count, dtype: int64
city
City 1    49
City 2    48
Name: count, dtype: int64
12
date          12
item           5
city           2
price         70
unit           1
data_label     1
dtype: int64


In [7]:
number = pd.to_numeric(raw["price"], errors="coerce")
print("missing before", raw["price"].isna().sum())
print("missing after", number.isna().sum())
raw[number.isna()][["date", "item", "city", "price"]]

missing before 1
missing after 2


,date,item,city,price
5,2026-01-01,Pulse C,City 2,NaN
17,2026-01-15,Staple A,City 2,"1,250"


In [8]:
number2 = pd.to_numeric(raw["price"].str.replace(",", ""), errors="coerce")
print(number2.describe())
raw[number2 > 1000][["date", "item", "city", "price"]]

count      96.000000
mean      315.458333
std      1007.561542
min        99.500000
25%       142.250000
50%       200.000000
75%       304.000000
max      9999.000000
Name: price, dtype: float64


,date,item,city,price
17,2026-01-15,Staple A,City 2,"1,250"
30,2026-01-22,Sweetener D,City 1,9999


In [9]:
print(raw[raw["item"] == "oil b"])
print(raw[(raw["date"] == "2026-02-05") & (raw["city"] == "City 1")][["item", "price"]])
print(raw[(raw["item"] == "Staple A") & (raw["city"] == "City 1")]["price"].tolist())

          date   item    city  price                       unit  \
40  2026-02-05  oil b  City 1  108.5  PKR per kg (illustrative)   

                                           data_label  
40  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...  
           item  price
40        oil b  108.5
42        Oil B  318.0
44      Pulse C  202.5
46  Sweetener D  161.5
['101.0', '99.5', '103.0', '106.5', '105.0', '107.0', '110.5', '114.0', '112.5', '116.0', '114.5', '99.5']


**Prompt sent to the AI tool:**
"I have a CSV with 97 rows and 6 columns: date (string, weekly, 12 values), item (string, 4 products),
city (string, 2 values), price (string, PKR per kg), unit (string, same in all rows), data_label (string, same in all rows).
Problems I found: one exact duplicate row, one item label in lowercase, one missing price, one price written with a comma
as '1,250' which makes the column text, one very large value, and one value 9999. Write pandas code to clean it."


In [10]:
ai = raw.copy()
ai["price"] = pd.to_numeric(ai["price"].str.replace(",", ""), errors="coerce")   
ai = ai.dropna(subset=["price"])                                                 
ai = ai.drop_duplicates()                                                        
ai["item"] = ai["item"].str.strip().str.title()                                  
q1, q3 = ai["price"].quantile(0.25), ai["price"].quantile(0.75)                  
iqr = q3 - q1
ai = ai[(ai["price"] >= q1 - 1.5 * iqr) & (ai["price"] <= q3 + 1.5 * iqr)]
ai = ai.drop(columns=["unit", "data_label"])                                     
ai["date"] = pd.to_datetime(ai["date"])                                          

In [11]:

t = raw.copy()
t["price"] = pd.to_numeric(t["price"].str.replace(",", ""), errors="coerce")
print("L1 missing:", t["price"].isna().sum(), "max:", t["price"].max())

a = t.dropna(subset=["price"])
print("L2 rows", len(t), "->", len(a))
print(t[t["price"].isna()][["date", "item", "city"]])      # the lost observation


b = a.drop_duplicates()
print("L3 rows", len(a), "->", len(b))


c = b.copy()
c["item"] = c["item"].str.strip().str.title()
print(c["item"].value_counts())
print("duplicate keys:", c.duplicated(subset=["date", "item", "city"]).sum())
print(c[c.duplicated(subset=["date", "item", "city"], keep=False)][["date", "item", "city", "price"]])


q1, q3 = c["price"].quantile(0.25), c["price"].quantile(0.75)
iqr = q3 - q1
e = c[(c["price"] >= q1 - 1.5 * iqr) & (c["price"] <= q3 + 1.5 * iqr)]
print("L5 rows", len(c), "->", len(e))
print(c[(c["price"] < q1 - 1.5 * iqr) | (c["price"] > q3 + 1.5 * iqr)][["date", "item", "city", "price"]])
print(e["item"].value_counts())


print("L6 columns left:", ai.columns.tolist())
print("L7 dtype:", ai["date"].dtype)

L1 missing: 1 max: 9999.0
L2 rows 97 -> 96
         date     item    city
5  2026-01-01  Pulse C  City 2
L3 rows 96 -> 95
item
Oil B          25
Sweetener D    24
Staple A       23
Pulse C        23
Name: count, dtype: int64
duplicate keys: 1
          date   item    city  price
40  2026-02-05  Oil B  City 1  108.5
42  2026-02-05  Oil B  City 1  318.0
L5 rows 95 -> 93
          date         item    city   price
17  2026-01-15     Staple A  City 2  1250.0
30  2026-01-22  Sweetener D  City 1  9999.0
item
Oil B          25
Pulse C        23
Sweetener D    23
Staple A       22
Name: count, dtype: int64
L6 columns left: ['date', 'item', 'city', 'price']
L7 dtype: datetime64[us]


In [12]:
clean = raw.copy()

clean = clean.drop_duplicates()

clean["flag"] = ""
clean.loc[40, "item"] = "Staple A"
clean.loc[40, "flag"] = "item relabelled oil b -> Staple A"
clean["item"] = clean["item"].str.strip().str.title()      


clean["price_raw"] = clean["price"]
clean["price"] = pd.to_numeric(clean["price"].str.replace(",", ""), errors="coerce")
clean.loc[clean["price_raw"].isna(), "flag"] = "price missing"
clean.loc[clean["price_raw"] == "1,250", "flag"] = "1,250 is 10x its neighbours"
clean.loc[clean["price_raw"] == "9999", "flag"] = "9999 sentinel"
clean.loc[clean["price"] > 1000, "price"] = float("nan")


clean["date"] = pd.to_datetime(clean["date"])

In [13]:
print(clean.shape)
print(clean["item"].value_counts())                                     
print(clean["city"].value_counts())                                     
print(clean["date"].nunique())                                          
print("duplicate keys:", clean.duplicated(subset=["date", "item", "city"]).sum())   
print("missing prices:", clean["price"].isna().sum())                   
clean[clean["flag"] != ""][["date", "item", "city", "price_raw", "price", "flag"]]

(96, 8)
item
Staple A       24
Oil B          24
Pulse C        24
Sweetener D    24
Name: count, dtype: int64
city
City 1    48
City 2    48
Name: count, dtype: int64
12
duplicate keys: 0
missing prices: 3


,date,item,city,price_raw,price,flag
5,2026-01-01,Pulse C,City 2,NaN,NaN,price missing
17,2026-01-15,Staple A,City 2,"1,250",NaN,"1,250 is 10x its neighbours"
30,2026-01-22,Sweetener D,City 1,9999,NaN,9999 sentinel
40,2026-02-05,Staple A,City 1,108.5,108.5,item relabelled oil b -> Staple A


In [14]:
print("BEFORE (comma removed, nothing else changed)")
print(number2.describe())
print("rows", len(raw), "| missing", number2.isna().sum(), "| item labels", raw["item"].nunique())

print("AFTER")
print(clean["price"].describe())
print("rows", len(clean), "| missing", clean["price"].isna().sum(), "| item labels", clean["item"].nunique())
clean.info()

BEFORE (comma removed, nothing else changed)
count      96.000000
mean      315.458333
std      1007.561542
min        99.500000
25%       142.250000
50%       200.000000
75%       304.000000
max      9999.000000
Name: price, dtype: float64
rows 97 | missing 1 | item labels 5
AFTER
count     93.000000
mean     203.607527
std       79.922070
min       99.500000
25%      149.000000
50%      200.000000
75%      298.000000
max      350.000000
Name: price, dtype: float64
rows 96 | missing 3 | item labels 4
<class 'pandas.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        96 non-null     datetime64[us]
 1   item        96 non-null     str           
 2   city        96 non-null     str           
 3   price       93 non-null     float64       
 4   unit        96 non-null     str           
 5   data_label  96 non-null     str           
 6   flag        9